# **Data Visualisation and Communication - CA2**

## **Index**

### **1. LIBRARIES**

In [1]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### **2. LOADING THE DATA**

In [2]:
## Changing the format file to a CSV
## Note: I had to replace the blank space between the words "Online" and "Retail" and add a "_" to make this code work.

FOLDER = r'C:\Users\felip\Documents\CA2-DataVisComms\DataVisComms-CA2' # This is the folder where the files will be.
XLSX_PATH = os.path.join(FOLDER, "Online_Retail.xlsx")
CSV_PATH = os.path.join(FOLDER, "Online_Retail.csv")

# Only convert if the CSV is not there yet
if not os.path.exists(CSV_PATH):
    print("( (!) CSV not found, converting the Excel file (this can take a few minutes)...")
    pd.read_excel(XLSX_PATH).to_csv(CSV_PATH, index=False)
    print("Done!")
else:
    print("======> CSV already exists, nothing to convert <======")

======> CSV already exists, nothing to convert <======


In [13]:
df = pd.read_csv(CSV_PATH, low_memory=False)

print("Rows:", df.shape[0], "| columns:", df.shape[1]) # printing how meny columns and rows.
df.head() # Checking the dataset

Rows: 541909 | columns: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [14]:
# Checking how many missing values the dataset has.
n = len(df)
miss = df.isnull().sum()
print(pd.DataFrame({"missing": miss, "pct": (miss / n* 100) .round (2)}))

             missing    pct
InvoiceNo          0   0.00
StockCode          0   0.00
Description     1454   0.27
Quantity           0   0.00
InvoiceDate        0   0.00
UnitPrice          0   0.00
CustomerID    135080  24.93
Country            0   0.00


In [15]:
# 1. Checking missing Descriptions
no_desc = df[df["Description"].isnull()]
print(no_desc["UnitPrice"].describe())

count    1454.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: UnitPrice, dtype: float64


In [16]:
print(no_desc["CustomerID"].describe())
# Observation: all 1,454 rows with missing Description have UnitPrice = 0 and no CustomerID
# This indicates non-sale records that can be removed.

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: CustomerID, dtype: float64


In [21]:
# 2. Filling from the same StockCode where possible
# This code will fill blank Descriptions by borrowing the description from the rows that have the same StockCode.
desc_map = (df.dropna(subset=["Description"])
            .groupby("StockCode")["Description"]
            .agg(lambda s: s.mode().iloc[0]))
df["Description"] = df["Description"].fillna(df["StockCode"].map(desc_map))
print("Still missing:", df["Description"].isnull().sum())
# output: Still missing: 112.
# This means 1,342 of the 1,454 blacn description were filled, and 112 could not be.

Still missing: 112


In [22]:
# 3. Dropping those 112 rows now.
before = len(df)
df = df.dropna(subset=["Description"])
print("Dropped:", before - len(df)) # this should print 112

df["is_guest"] = df["CustomerID"].isnull()
print(df["is_guest"].sum(), "guest rows kepts")

Dropped: 112
134968 guest rows kepts


In [25]:
# Checking the missing values again.
n = len(df)
miss = df.isnull().sum()
print(pd.DataFrame({"missing": miss, "pct": (miss / n * 100).round(2)}))
print("Rows:", n)
# Observation: the missing values for CustomerID are still.
# Imputing an ID would merge unrelated buyers into one fake customer, and that would distort customer
# counts, repeat-purchase rates, etc.

             missing    pct
InvoiceNo          0   0.00
StockCode          0   0.00
Description        0   0.00
Quantity           0   0.00
InvoiceDate        0   0.00
UnitPrice          0   0.00
CustomerID    134968  24.91
Country            0   0.00
is_guest           0   0.00
Rows: 541797
